# Tests for `newrelic_sb_sdk.shortcuts.historical_data_export`

## Imports

In [ ]:
import warnings
from unittest.mock import MagicMock, patch

import pytest

In [ ]:
from newrelic_sb_sdk.client import NewRelicClient
from newrelic_sb_sdk.graphql.enums import HistoricalDataExportStatus
from newrelic_sb_sdk.graphql.objects import (
    Account,
    HistoricalDataExportCustomerExportResponse,
)
from newrelic_sb_sdk.graphql.scalars import Nrql
from newrelic_sb_sdk.shortcuts.historical_data_export import (
    _perform_historical_data_export,
    can_execute_historical_data_export,
    cancel_historical_data_export,
    create_historical_data_export,
    get_all_historical_data_exports,
    get_historical_data_export,
    perform_historical_data_export,
)
from newrelic_sb_sdk.utils.exceptions import NewRelicError

## Tests

In [ ]:
class TestShortcutsHistoricalDataExport:
    @pytest.fixture
    def client(self):
        return NewRelicClient(new_relic_user_key="NRAK-123456789012345678901234567")

    @pytest.fixture
    def account(self):
        return Account(json_data={"id": 123, "name": "Acme Corp"})

    @pytest.fixture
    def base_response(self):
        return {
            "id": "123-export",
            "status": "COMPLETE_SUCCESS",
            "percentComplete": 100.0,
            "results": ["http://example.com/data"],
            "message": "",
        }

    @patch("newrelic_sb_sdk.shortcuts.historical_data_export.NewRelicClient.execute")
    def test_create_historical_data_export(
        self, mock_execute, client, account, base_response
    ):
        mock_response = MagicMock()
        mock_response.json.return_value = {
            "data": {"historicalDataExportCreateExport": base_response}
        }
        mock_execute.return_value = mock_response

        result = create_historical_data_export(
            client=client,
            account=account,
            nrql_query=Nrql("SELECT count(*) FROM Transaction"),
        )

        assert isinstance(result, HistoricalDataExportCustomerExportResponse)
        assert result.id == "123-export"

    @patch("newrelic_sb_sdk.shortcuts.historical_data_export.NewRelicClient.execute")
    def test_get_all_historical_data_exports(
        self, mock_execute, client, account, base_response
    ):
        mock_response = MagicMock()
        mock_response.json.return_value = {
            "data": {
                "actor": {
                    "account": {"historicalDataExport": {"exports": [base_response]}}
                }
            }
        }
        mock_execute.return_value = mock_response

        results = get_all_historical_data_exports(client=client, account=account)

        assert len(results) == 1
        assert isinstance(results[0], HistoricalDataExportCustomerExportResponse)
        assert results[0].id == "123-export"

    @patch("newrelic_sb_sdk.shortcuts.historical_data_export.NewRelicClient.execute")
    def test_get_historical_data_export(
        self, mock_execute, client, account, base_response
    ):
        mock_response = MagicMock()
        mock_response.json.return_value = {
            "data": {
                "actor": {
                    "account": {"historicalDataExport": {"export": base_response}}
                }
            }
        }
        mock_execute.return_value = mock_response

        result = get_historical_data_export(
            client=client, account=account, export_id="123-export"
        )

        assert isinstance(result, HistoricalDataExportCustomerExportResponse)
        assert result.id == "123-export"

    @patch("newrelic_sb_sdk.shortcuts.historical_data_export.NewRelicClient.execute")
    def test_cancel_historical_data_export(
        self, mock_execute, client, account, base_response
    ):
        base_response["status"] = "CANCELED"
        mock_response = MagicMock()
        mock_response.json.return_value = {
            "data": {"historicalDataExportCancelExport": base_response}
        }
        mock_execute.return_value = mock_response

        result = cancel_historical_data_export(
            client=client, account=account, export_id="123-export"
        )

        assert isinstance(result, HistoricalDataExportCustomerExportResponse)
        assert result.status == HistoricalDataExportStatus("CANCELED")

    @patch(
        "newrelic_sb_sdk.shortcuts.historical_data_export.get_all_historical_data_exports"
    )
    def test_can_execute_historical_data_export_true(
        self, mock_get_all, client, account, base_response
    ):
        base_response["status"] = "COMPLETE_SUCCESS"
        mock_get_all.return_value = [
            HistoricalDataExportCustomerExportResponse(json_data=base_response)
        ]

        assert (
            can_execute_historical_data_export(client=client, account=account) is True
        )

    @patch(
        "newrelic_sb_sdk.shortcuts.historical_data_export.get_all_historical_data_exports"
    )
    def test_can_execute_historical_data_export_false(
        self, mock_get_all, client, account, base_response
    ):
        r1 = base_response.copy()
        r1["status"] = "WAITING"
        r2 = base_response.copy()
        r2["status"] = "IN_PROGRESS"

        mock_get_all.return_value = [
            HistoricalDataExportCustomerExportResponse(json_data=r1),
            HistoricalDataExportCustomerExportResponse(json_data=r2),
        ]

        assert (
            can_execute_historical_data_export(client=client, account=account) is False
        )

    @patch("time.sleep", return_value=None)
    @patch(
        "newrelic_sb_sdk.shortcuts.historical_data_export.can_execute_historical_data_export",
        return_value=True,
    )
    @patch(
        "newrelic_sb_sdk.shortcuts.historical_data_export.create_historical_data_export"
    )
    @patch(
        "newrelic_sb_sdk.shortcuts.historical_data_export.get_historical_data_export"
    )
    def test_perform_historical_data_export_success(
        self,
        mock_get,
        mock_create,
        mock_can,
        mock_sleep,
        client,
        account,
        base_response,
    ):
        waiting_resp = base_response.copy()
        waiting_resp["status"] = "IN_PROGRESS"

        mock_create.return_value = HistoricalDataExportCustomerExportResponse(
            json_data=waiting_resp
        )
        mock_get.side_effect = [
            HistoricalDataExportCustomerExportResponse(json_data=waiting_resp),
            HistoricalDataExportCustomerExportResponse(json_data=base_response),
        ]

        result = _perform_historical_data_export(
            client=client,
            account=account,
            nrql_query=Nrql("SELECT count(*) FROM Transaction"),
        )

        assert result.status == HistoricalDataExportStatus("COMPLETE_SUCCESS")
        assert mock_get.call_count == 2

    @patch("time.sleep", return_value=None)
    @patch(
        "newrelic_sb_sdk.shortcuts.historical_data_export.cancel_historical_data_export"
    )
    @patch(
        "newrelic_sb_sdk.shortcuts.historical_data_export.can_execute_historical_data_export",
        return_value=True,
    )
    @patch(
        "newrelic_sb_sdk.shortcuts.historical_data_export.create_historical_data_export"
    )
    @patch(
        "newrelic_sb_sdk.shortcuts.historical_data_export.get_historical_data_export"
    )
    def test_perform_historical_data_export_failed(
        self,
        mock_get,
        mock_create,
        mock_can,
        mock_cancel,
        mock_sleep,
        client,
        account,
        base_response,
    ):
        fail_resp = base_response.copy()
        fail_resp["status"] = "COMPLETE_FAILED"

        mock_create.return_value = HistoricalDataExportCustomerExportResponse(
            json_data=fail_resp
        )
        mock_get.return_value = HistoricalDataExportCustomerExportResponse(
            json_data=fail_resp
        )

        with pytest.raises(NewRelicError):
            _perform_historical_data_export(
                client=client,
                account=account,
                nrql_query=Nrql("SELECT count(*) FROM Transaction"),
            )

        mock_cancel.assert_called()

    @patch(
        "newrelic_sb_sdk.shortcuts.historical_data_export.cancel_historical_data_export"
    )
    @patch("newrelic_sb_sdk.shortcuts.historical_data_export.download_files")
    @patch(
        "newrelic_sb_sdk.shortcuts.historical_data_export._perform_historical_data_export"
    )
    def test_perform_historical_data_export_wrapper(
        self, mock_perform, mock_download, mock_cancel, client, account, base_response
    ):
        mock_perform.return_value = HistoricalDataExportCustomerExportResponse(
            json_data=base_response
        )

        perform_historical_data_export(
            client=client,
            account=account,
            nrql_query=Nrql("SELECT count(*) FROM Transaction"),
            base_file_name="test_export",
        )

        mock_download.assert_called_once_with(
            urls=["http://example.com/data"],
            base_file_name="test_export",
            file_extension="json.gz",
        )
        mock_cancel.assert_called_once_with(
            client=client, account=account, export_id="123-export"
        )